### Supplement 1: Prompt chaining (`1-prompt-chaining.py`)

Turn a sentence like "Let's schedule a meeting next Tuesday" into a confirmation message, using three small LLM calls in a row.

**The key point:** Your code runs the steps in a fixed order, and each step is one LLM call that returns a Pydantic object. A plain `if` after step 1, the gate, stops the chain when the text isn't a calendar event. The model never decides what runs next; that is the difference from tool calling in [Supplement_3-tools.ipynb](../1-introduction/Supplement_3-tools.ipynb).

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
user_input
   │
   ▼
Step 1   extract_event_info      → EventExtraction
   │     Is this a calendar event? How sure is the model?
   ▼
Gate     Not an event, or confidence below 0.7 → stop, return None
   │
   ▼
Step 2   parse_event_details     → EventDetails
   │     Name, date, duration, participants
   ▼
Step 3   generate_confirmation   → EventConfirmation
         The message the user reads
</pre>

#### 1. Setup
`load_dotenv()` copies `OPENAI_API_KEY` from `.env` into `os.environ`, and `OpenAI()` reads it from there. `model` is stored once, so all three steps use the same model.

In [1]:
from typing import Optional
from datetime import datetime
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
model = "gpt-5-nano"

#### 2. One output shape per step
Each class is the `response_format` of one step, so that step's reply comes back as an object of the class. The `Field(description=...)` texts are sent to the model as part of the schema, the same as in [Supplement_2-structured.ipynb](../1-introduction/Supplement_2-structured.ipynb).
- `EventExtraction`: The gate reads its `is_calendar_event` and `confidence_score`.
- `EventDetails`: `date` is asked for in ISO 8601, a standard format like `2026-10-27T14:00:00`.
- `EventConfirmation`: `calendar_link` is `Optional[str]`, so the model may send `None` for it.

In [2]:
# Define the data models for each stage

class EventExtraction(BaseModel):
    """First LLM call: Extract basic event information"""

    description: str = Field(description="Raw description of the event")
    is_calendar_event: bool = Field(
        description="Whether this text describes a calendar event"
    )
    confidence_score: float = Field(description="Confidence score between 0 and 1")


class EventDetails(BaseModel):
    """Second LLM call: Parse specific event details"""

    name: str = Field(description="Name of the event")
    date: str = Field(
        description="Date and time of the event. Use ISO 8601 to format this value."
    )
    duration_minutes: int = Field(description="Expected duration in minutes")
    participants: list[str] = Field(description="List of participants")


class EventConfirmation(BaseModel):
    """Third LLM call: Generate confirmation message"""

    confirmation_message: str = Field(
        description="Natural language confirmation message"
    )
    calendar_link: Optional[str] = Field(
        description="Generated calendar link if applicable"
    )

#### 3. Step 1: Is it a calendar event?
`extract_event_info` puts today's date in the system message and asks the model to fill an `EventExtraction`. It is an ordinary `parse()` call wrapped in a function. The cells after it run step 1 once on its own, so you can see what it returns.

In [3]:
# Define the functions

def extract_event_info(user_input: str) -> EventExtraction:
    """First LLM call to determine if input is a calendar event"""
    print("Starting event extraction analysis")
    print(f"Input text: {user_input}")

    today = datetime.now()
    date_context = f"Today is {today.strftime('%A, %B %d, %Y')}."

    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"{date_context} Analyze if the text describes a calendar event.",
            },
            {"role": "user", "content": user_input},
        ],
        response_format=EventExtraction,
    )
    result = completion.choices[0].message.parsed
    print(
        f"Extraction complete - Is calendar event: {result.is_calendar_event}, Confidence: {result.confidence_score:.2f}"
    )
    return result

In [4]:
user_input = "Let's schedule a 1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap."

extraction = extract_event_info(user_input)

Starting event extraction analysis
Input text: Let's schedule a 1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap.
Extraction complete - Is calendar event: True, Confidence: 0.92


In [5]:
print(type(extraction))
print(extraction.model_dump_json(indent=2))

<class '__main__.EventExtraction'>
{
  "description": "1-hour team meeting on Tue Oct 27, 2026 at 2:00 PM with Alice and Bob to discuss the project roadmap.",
  "is_calendar_event": true,
  "confidence_score": 0.92
}


#### 4. Step 2: Pull out the details
`parse_event_details` receives only the `description` from step 1, not the original sentence. It also gets today's date, so the model can turn "next Tuesday" into a real date. Step 1's model writes that `description` itself, so it can be a rewording rather than a copy: In the output above it already says "Tue Oct 27, 2026".

In [6]:
def parse_event_details(description: str) -> EventDetails:
    """Second LLM call to extract specific event details"""
    print("Starting event details parsing")

    today = datetime.now()
    date_context = f"Today is {today.strftime('%A, %B %d, %Y')}."

    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"{date_context} Extract detailed event information. When dates reference 'next Tuesday' or similar relative dates, use this current date as reference.",
            },
            {"role": "user", "content": description},
        ],
        response_format=EventDetails,
    )
    result = completion.choices[0].message.parsed
    print(
        f"Parsed event details - Name: {result.name}, Date: {result.date}, Duration: {result.duration_minutes}min"
    )
    print(f"Participants: {', '.join(result.participants)}")
    return result

In [7]:
details = parse_event_details(extraction.description)

Starting event details parsing
Parsed event details - Name: Team meeting: project roadmap discussion, Date: 2026-10-27T14:00:00, Duration: 60min
Participants: Alice, Bob


In [8]:
print(type(details))
print(details.model_dump_json(indent=2))

<class '__main__.EventDetails'>
{
  "name": "Team meeting: project roadmap discussion",
  "date": "2026-10-27T14:00:00",
  "duration_minutes": 60,
  "participants": [
    "Alice",
    "Bob"
  ]
}


#### 5. Step 3: Write the confirmation
`generate_confirmation` sends the `EventDetails` object as text: `model_dump()` turns it into a dict, and `str()` turns that dict into a string, because a message's `content` must be text.

In [9]:
def generate_confirmation(event_details: EventDetails) -> EventConfirmation:
    """Third LLM call to generate a confirmation message"""
    print("Generating confirmation message")

    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Generate a natural confirmation message for the event. Sign off with your name; Susie",
            },
            {"role": "user", "content": str(event_details.model_dump())},
        ],
        response_format=EventConfirmation,
    )
    result = completion.choices[0].message.parsed
    print("Confirmation message generated successfully")
    return result

In [10]:
confirmation = generate_confirmation(details)

Generating confirmation message
Confirmation message generated successfully


In [11]:
print(type(confirmation))
print(confirmation.model_dump_json(indent=2))

<class '__main__.EventConfirmation'>
{
  "confirmation_message": "Hi Alice and Bob,\n\nThis is a confirmation for the Team meeting: project roadmap discussion on October 27, 2026 at 2:00 PM for 60 minutes.\nAttendees: Alice, Bob.\n\nAgenda (tentative): review current progress, align on milestones, and outline next steps for the project roadmap.\n\nIf this time works for you, I look forward to our discussion. If not, please let me know and we can adjust.\n\nBest regards,\nSusie",
  "calendar_link": null
}


#### 6. The chain, with the gate
`process_calendar_request` does what sections 3 to 5 just did by hand, plus the gate: If step 1 says it isn't an event, or its `confidence_score` is below 0.7, it returns `None`, and steps 2 and 3 never run. The score is the model's own guess, not a measured probability.

In [12]:
# Chain the functions together

def process_calendar_request(user_input: str) -> Optional[EventConfirmation]:
    """Main function implementing the prompt chain with gate check"""
    print("Processing calendar request")
    print(f"Raw input: {user_input}")

    # First LLM call: Extract basic info
    initial_extraction = extract_event_info(user_input)

    # Gate check: Verify if it's a calendar event with sufficient confidence
    if (
        not initial_extraction.is_calendar_event
        or initial_extraction.confidence_score < 0.7
    ):
        print(
            f"Gate check failed - is_calendar_event: {initial_extraction.is_calendar_event}, confidence: {initial_extraction.confidence_score:.2f}"
        )
        return None

    print("Gate check passed, proceeding with event processing")

    # Second LLM call: Get detailed event information
    event_details = parse_event_details(initial_extraction.description)

    # Third LLM call: Generate confirmation
    confirmation = generate_confirmation(event_details)

    print("Calendar request processing completed successfully")
    return confirmation

#### 7. A calendar request: All three steps run

In [13]:
# Test the chain with a valid input

user_input = "Let's schedule a 1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap."

result = process_calendar_request(user_input)
if result:
    print(f"Confirmation: {result.confirmation_message}")
    if result.calendar_link:
        print(f"Calendar Link: {result.calendar_link}")
else:
    print("This doesn't appear to be a calendar event request.")

Processing calendar request
Raw input: Let's schedule a 1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap.
Starting event extraction analysis
Input text: Let's schedule a 1h team meeting next Tuesday at 2pm with Alice and Bob to discuss the project roadmap.
Extraction complete - Is calendar event: True, Confidence: 0.92
Gate check passed, proceeding with event processing
Starting event details parsing
Parsed event details - Name: Team Meeting: Project Roadmap, Date: 2026-10-27T14:00:00, Duration: 60min
Participants: Alice, Bob
Generating confirmation message
Confirmation message generated successfully
Calendar request processing completed successfully
Confirmation: Hi Alice and Bob,

This is a confirmation for the Team Meeting: Project Roadmap.

Date and time: 27 October 2026 at 2:00 PM (local time)
Duration: 60 minutes
Participants: Alice, Bob

Agenda:
- Review current project milestones
- Align on upcoming deliverables and owners
- Identify blocker

#### 8. Not a calendar request: The gate stops it
Step 1 runs, the gate fails, and the function returns `None`. Only one API call is made.

In [14]:
# Test the chain with an invalid input

user_input = "Can you send an email to Alice and Bob to discuss the project roadmap?"

result = process_calendar_request(user_input)
if result:
    print(f"Confirmation: {result.confirmation_message}")
    if result.calendar_link:
        print(f"Calendar Link: {result.calendar_link}")
else:
    print("This doesn't appear to be a calendar event request.")

Processing calendar request
Raw input: Can you send an email to Alice and Bob to discuss the project roadmap?
Starting event extraction analysis
Input text: Can you send an email to Alice and Bob to discuss the project roadmap?
Extraction complete - Is calendar event: False, Confidence: 0.85
Gate check failed - is_calendar_event: False, confidence: 0.85
This doesn't appear to be a calendar event request.
